<h1>
  <left>
    <font color="plum">CSAI 463 - Computational Intelligence</font><br>
    <font color="white">P300 Speller BCI System</font>
  </left>
</h1>

In [1]:
# Importing Our Libraries
import os
import numpy as np
import tkinter as tk
from scipy.signal import butter, lfilter
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA

In [3]:
dataset_dir = r"C:\NGU\Semester 6\Computational Intellegence\Project\dataset"

# Load the training EEG signals (shape: channels x time) for Subject A
train_signal = np.loadtxt(os.path.join(dataset_dir, "Subject_A_Train_Signal.txt")) 

# Load the flashing matrix indicating which stimulus was flashed at each time point during training
train_flashing = np.loadtxt(os.path.join(dataset_dir, "Subject_A_Train_Flashing.txt")) 

# Load the stimulus code matrix (which row/column was flashed) for the training session
train_stimulus_code = np.loadtxt(os.path.join(dataset_dir, "Subject_A_Train_StimulusCode.txt")) 

# Load the stimulus type matrix (target or non-target) for the training session
train_stimulus_type = np.loadtxt(os.path.join(dataset_dir, "Subject_A_Train_StimulusType.txt")) 

# Read the sequence of characters that the subject was focusing on during the training session
train_target_char = open(os.path.join(dataset_dir, "Subject_A_Train_TargetChar.txt")).read().strip() 

# Load the testing EEG signals for Subject A (to be used for prediction)
test_signal = np.loadtxt(os.path.join(dataset_dir, "Subject_A_Test_Signal.txt"))  

test_flashing = np.loadtxt(os.path.join(dataset_dir, "Subject_A_Test_Flashing.txt")) 
test_stimulus_code = np.loadtxt(os.path.join(dataset_dir, "Subject_A_Test_StimulusCode.txt"))  

In [5]:
lowcut = 1.0         # Lower cutoff frequency to remove slow drifts
highcut = 15.0       # Upper cutoff frequency to remove muscle noise and high-frequency components

fs = 240 

# Order of the Butterworth filter (higher order = steeper cutoff)
order = 4  

# Design a Butterworth bandpass filter

def butter_bandpass(lowcut, highcut, fs, order = 4):
    nyq = 0.5 * fs  
    low = lowcut / nyq  
    high = highcut / nyq  
    b, a = butter(order, [low, high], btype = 'band')  
    return b, a

# Apply the bandpass filter to multichannel EEG data

def bandpass_filter(data, lowcut, highcut, fs, order = 4):
    b, a = butter_bandpass(lowcut, highcut, fs, order = order) 
    filtered_data = lfilter(b, a, data, axis = 1)  
    return filtered_data

# Training
train_signal_filtered = bandpass_filter(train_signal, lowcut, highcut, fs, order) 

# Testing
test_signal_filtered = bandpass_filter(test_signal, lowcut, highcut, fs, order)  

print("Train and Test Filtered EEG Signals are Ready.")

Train and Test Filtered EEG Signals are Ready.


In [6]:
# Extract EEG epochs for debugging purposes with a cap on memory usage

def extract_epochs_debug(signal, flashing, stimulus_code, window=(-0.2, 0.8), fs=240, max_epochs=1000):
    """
    Extract epochs from EEG signal with reduced memory load for debugging.

    Args:
    - signal: EEG signal matrix (channels x samples)
    - flashing: Matrix indicating when each flash occurred
    - stimulus_code: Matrix indicating which row/column was flashed
    - window: Time window for epoch extraction in seconds (start, end)
    - fs: Sampling frequency of the EEG signal
    - max_epochs: Maximum number of epochs to extract (to limit memory usage)

    Returns:
    - epochs: Array of extracted EEG epochs (n_epochs x channels x timepoints)
    - labels: Array of corresponding stimulus codes for each epoch
    """

    # Get number of EEG channels and total number of samples
    n_channels, n_samples = signal.shape

    # Convert time window to sample indices for epoch extraction
    epoch_start = int(window[0] * fs)
    epoch_end = int(window[1] * fs)
    epoch_length = epoch_end - epoch_start

    sample_limit = min(n_samples, 3000)
    flashing = flashing[:sample_limit]
    stimulus_code = stimulus_code[:sample_limit]
    signal = signal[:, :sample_limit]

    # Find the timepoints (sample, index) where flashing occurred (i.e., value == 1)
    flash_samples, flash_indices = np.where(flashing == 1)

    # Preallocate memory
    epochs = np.zeros((max_epochs, n_channels, epoch_length), dtype=np.float32)
    labels = np.zeros((max_epochs,), dtype=np.int32)

    # Extract segments of EEG around each flash
    count = 0
    for sample, flash in zip(flash_samples, flash_indices):
        start = sample + epoch_start
        end = sample + epoch_end
        if start >= 0 and end < sample_limit:
            epochs[count] = signal[:, start:end].astype(np.float32)
            labels[count] = stimulus_code[sample, flash]
            count += 1
            if count >= max_epochs:
                break

    return epochs[:count], labels[:count]       # Return only the filled part of the arrays


# Apply the epoch extraction on training data (limited to 2000 epochs)
train_epochs_debug, train_labels_debug = extract_epochs_debug(
    train_signal_filtered[:64, :], train_flashing, train_stimulus_code, max_epochs=2000
)
print(f"Successfuly Extracted {len(train_epochs_debug)} Training Epochs, Shape: {train_epochs_debug[0].shape}")

test_epochs_debug, test_labels_debug = extract_epochs_debug(
    test_signal_filtered[:64, :], test_flashing, test_stimulus_code, max_epochs=2000
)
print(f"Successfuly Extracted {len(test_epochs_debug)} Test Epochs, Shape: {test_epochs_debug[0].shape}")

Successfuly Extracted 2000 Training Epochs, Shape: (64, 240)
Successfuly Extracted 2000 Test Epochs, Shape: (64, 240)


In [7]:
class CSP(BaseEstimator, TransformerMixin):
    def __init__(self, n_components = 4):
        self.n_components = n_components  # Number of spatial filters (components) to keep

        
    def fit(self, X, y):
        covs = [np.cov(x) for x in X]    # Compute covariance matrices for each EEG epoch

        cov_avg = np.mean(covs, axis = 0)  # Compute average covariance matrix across all epochs

        eigvals, eigvecs = np.linalg.eigh(cov_avg)  # Perform eigenvalue decomposition on the average covariance matrix

        idx = np.argsort(eigvals)[::-1]  # Sort eigenvectors by decreasing eigenvalue magnitude

        self.filters_ = eigvecs[:, idx[:self.n_components]]  # Select the top n_components eigenvectors as spatial filters

        return self

    def transform(self, X):
        X_transformed = [np.dot(self.filters_.T, x) for x in X] 
        return np.array([x.flatten() for x in X_transformed]) 


csp = CSP(n_components = 4)
lda = LDA()


subset_size = 1000
train_epochs_subset = train_epochs_debug[:subset_size]
train_labels_subset = train_labels_debug[:subset_size]

# Convert labels to binary: 1 if stimulus code is 1 (target), else 0 (non-target)

train_labels_binary_subset = (train_labels_subset == 1).astype(int)

# Fit the CSP transformer to the EEG training data using binary labels

csp.fit(train_epochs_subset, train_labels_binary_subset)
X_train_csp = csp.transform(train_epochs_subset)

lda.fit(X_train_csp, train_labels_binary_subset)

print(f"The CSP and LDA Successfully Trained on {subset_size} Epochs.") 

The CSP and LDA Successfully Trained on 1000 Epochs.


In [8]:
batch_size = 10000 
n_test_epochs = len(test_epochs_debug)  # Get the total number of test epochs

test_predictions = [] 

# Process the test data in batches
for start_idx in range(0, n_test_epochs, batch_size):
    end_idx = min(start_idx + batch_size, n_test_epochs)  # don't exceed total

    batch_epochs = test_epochs_debug[start_idx:end_idx]  

    
    X_test_csp = csp.transform(batch_epochs) # Transform the batch using the trained CSP filters


   # Predict probabilities using the trained LDA classifier
    # [:, 1] selects the probability of class 1 (target stimulus)
    probas = lda.predict_proba(X_test_csp)[:, 1]
    test_predictions.extend(probas)  

    print(f"Finished Processing {end_idx}/{n_test_epochs} Test Epochs.")
    
# Convert predictions list to a NumPy array
test_predictions = np.array(test_predictions)
print("Test Predictions Successfully Completed.")  

Finished Processing 2000/2000 Test Epochs.
Test Predictions Successfully Completed.


In [9]:
# Define the P300 speller matrix (6x6 grid of characters)

speller_matrix = [
    ['A', 'B', 'C', 'D', 'E', 'F'],
    ['G', 'H', 'I', 'J', 'K', 'L'],
    ['M', 'N', 'O', 'P', 'Q', 'R'],
    ['S', 'T', 'U', 'V', 'W', 'X'],
    ['Y', 'Z', '1', '2', '3', '4'],
    ['5', '6', '7', '8', '9', '0']
]
# Decode predicted probabilities into characters using the speller matrix

def decode_letters(predictions, flashes_per_letter = 12): # Calculate how many letters are present in the prediction set

    n_letters = len(predictions) // flashes_per_letter 
    decoded_sentence = [] 

    for i in range(n_letters):  # Loop over each letter's prediction group

        # Extract the 12 predictions corresponding to one character (6 rows + 6 columns)
        letter_preds = predictions[i * flashes_per_letter:(i + 1) * flashes_per_letter]
        row_preds = letter_preds[:6]  
        col_preds = letter_preds[6:]  

        # Find the row and column with the highest prediction probability
        row_idx = np.argmax(row_preds)
        col_idx = np.argmax(col_preds)

        decoded_char = speller_matrix[row_idx][col_idx] # Use the indices to look up the predicted character in the speller matrix

        decoded_sentence.append(decoded_char) 
    
    return ''.join(decoded_sentence)   # Join the list of characters into a full decoded sentence string


decoded_sentence = decode_letters(test_predictions) # Decode the predicted test probabilities into characters

print("Your Final Decoded Sentence is:", decoded_sentence)  

Your Final Decoded Sentence is: AAAAAAAAYAAAAAAACAAAAAAAAAAAAAAAAYAAAAAAACAAAAAAAAAAAAAAAAYAAAAAAACAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAYAAAAAAA


In [ ]:
output_path = r"C:\NGU\Semester 6\Computational Intellegence\Project\dataset\output_real.txt"

with open(output_path, 'w') as f:
    f.write(decoded_sentence)

print(f"Decoded Sentence Saved to {output_path}")

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\NGU\\Semester 6\\Computational Intellegence\\Final Project\\dataset\\output_real.txt'

In [12]:
root = tk.Tk()
root.title("CI Final Project - OUR P300 SPELLER BCI System")

speller_matrix = [
    ['A', 'B', 'C', 'D', 'E', 'F'],
    ['G', 'H', 'I', 'J', 'K', 'L'],
    ['M', 'N', 'O', 'P', 'Q', 'R'],
    ['S', 'T', 'U', 'V', 'W', 'X'],
    ['Y', 'Z', '1', '2', '3', '4'],
    ['5', '6', '7', '8', '9', '0']
]

# Create label widgets for each character in the matrix and arrange them in a grid

labels = []
for i in range(6):
    row = []
    for j in range(6):
        label = tk.Label(root, text = speller_matrix[i][j], width = 6, height = 3, borderwidth = 2, relief = "solid", font = ('Arial', 24))
        label.grid(row = i, column = j)
        row.append(label)
    labels.append(row)

# Create a label to display the final decoded sentence dynamically

sentence_label = tk.Label(root, text = "Your Final Decoded Sentence: ", font = ('Arial', 18))
sentence_label.grid(row = 6, column = 0, columnspan = 6)

decoded_sentence_gui = []  # Holds the decoded sentence
current_letter_index = 0   # Tracks which letter we're currently flashing for
flashes_per_letter = 12  

# simulate flashing and decoding one letter

def flash_simulation():
    global current_letter_index
    if current_letter_index * flashes_per_letter >= len(test_predictions):  # Stop flashing when all letters have been decoded

        sentence_label.config(text = "Your Final Decoded Sentence: " + ''.join(decoded_sentence_gui) + " (End of Message)")
        return

    row_sums = np.zeros(6)
    col_sums = np.zeros(6)
    row_counts = np.zeros(6)
    col_counts = np.zeros(6)
 
    for flash_num in range(12):
        flash_idx = current_letter_index * flashes_per_letter + flash_num
        prob = test_predictions[flash_idx]
        if flash_num < 6:
            row_idx = flash_num  
            for j in range(6):
                labels[row_idx][j].config(bg = 'blue')
            root.update()
            root.after(200)
            for j in range(6):
                labels[row_idx][j].config(bg = 'SystemButtonFace')
            row_sums[row_idx] += prob
            row_counts[row_idx] += 1
        else:

            col_idx = flash_num - 6  
            for i in range(6):
                labels[i][col_idx].config(bg = 'blue')
            root.update()
            root.after(200)  # Wait for 200 ms
            for i in range(6):
                labels[i][col_idx].config(bg = 'SystemButtonFace')
            col_sums[col_idx] += prob
            col_counts[col_idx] += 1
        root.after(100)   # Brief pause between flashes

    # Compute average probabilities for each row and column

    row_avgs = row_sums / np.maximum(row_counts, 1)
    col_avgs = col_sums / np.maximum(col_counts, 1)

     # Select the row and column with the highest average probability

    selected_row = int(np.argmax(row_avgs))
    selected_col = int(np.argmax(col_avgs))

     # Decode the character from the speller matrix

    decoded_char = speller_matrix[selected_row][selected_col]
    decoded_sentence_gui.append(decoded_char)

    sentence_label.config(text = f"Final Decoded Sentence: {''.join(decoded_sentence_gui)}")

     # Move to the next letter

    current_letter_index += 1
root.bind('<space>', lambda event: flash_simulation()) # Bind the spacebar key to trigger a flash simulation (one letter at a time)


root.mainloop()